# ChIPDiff reproduction- H3K27me3 ESC vs NPC

This notebook reproduces the preprocessing and bin-level analysis described by Xu et al. (2008) for the comparaison of H3K27me3 profiles between mouse embryonic stem cells (ESC) and neural progenitor cells (NPC). 

The workflow starts from aligned single-end-ChIP-seq reads and progressively transforms them into genomic regions that can later be classified as differentially modified. 

## Workflow
1. Load and inspect the aligned ChIP-seq reads
2. Remove redundant tags potentially produced by PCR amplification
3. Estimate the center of each ChIP fragment
4. Assign fragment centers to 1kb genomic bins
5. Couny ESC and NPC fragments in each bin
6. Retain bins with sufficant combined H3K27me3 signal
7. Estimate modification intensities probabilistically
8. Use an HMM to classify bins as:
    - non-differential
    - ESC-enriched
    - NPC-enriched
9. Merge adjacent differential bins into DHMS regions

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import math

### 1 Load and check the aligned ChIP-seq reads

The analysis starts form single-end ChIP-seq reads alredy aligned to the mouse **mm8** referece genome. Each aligned read, called a *tag* in the original paper, represents one sequenced end of a larger ChIP DNA fragment. 

In [2]:
esc_path = Path("../data/raw/GSM307619_ES.H3K27me3.aligned.txt.gz")

npc_path = Path("../data/raw/GSM307614_NP.H3K27me3.aligned.txt.gz")

columns=["chromosome","start","end","orientation","id","mismatch","sequence"]

In [3]:
def create_df(path):
    df_path= pd.read_csv(path,sep='\t',names=columns)
    print(f"Shape: {df_path.shape}")
    print()
    print(df_path.dtypes)
    print()
    display(df_path.head())
    return df_path

esc_df=create_df(esc_path)

Shape: (6537926, 7)

chromosome       str
start          int64
end            int64
orientation      str
id               str
mismatch       int64
sequence         str
dtype: object



,chromosome,start,end,orientation,id,mismatch,sequence
0,chr5,35526975,35527007,-,3481.6.1,1,GCAATAACTTAAGTTCATTATAATCCATTAAA
1,chr6,99092111,99092143,+,3481.6.2,1,GAATATGGGAAGCCTGCAGCAACAGGCTCATT
2,chr19,49690961,49690993,-,3481.6.3,4,GCTATTGATGAGTGTGTTGAGGGCAACCTAAC
3,chr19,56008355,56008387,-,3481.6.4,2,GGTCTCCGCATAGGTATGGCTCACCGCGGTTG
4,chr2,121808458,121808490,+,3481.6.5,1,GAGGGTTGGGAACTCACAGGCATCCTTGGGCC


In [4]:
npc_df = create_df(npc_path)

Shape: (7949806, 7)

chromosome       str
start          int64
end            int64
orientation      str
id               str
mismatch       int64
sequence         str
dtype: object



,chromosome,start,end,orientation,id,mismatch,sequence
0,chr9,104512134,104512161,-,3099.6.1,0,GAATTATAACTGGGGAGGGGAGGGATA
1,chr15,72348592,72348619,+,3099.6.2,0,GGATGGGATGAGGCCATCACAAATCAG
2,chr1,164198794,164198821,-,3099.6.3,0,GTTCTCCCCTACTGGTCAAAATGACAG
3,chr18,11264277,11264304,-,3099.6.4,1,GTGTGAAGGTCTCCTGAACATGATTCT
4,chr4,54494488,54494515,-,3099.6.5,0,GTGCAATCTTAAATGATTAATTTATCC


In [5]:
chromosomes= np.unique(npc_df['chromosome'])
print(chromosomes)
print(len(chromosomes))

['chr1' 'chr10' 'chr11' 'chr12' 'chr13' 'chr14' 'chr15' 'chr16' 'chr17'
 'chr18' 'chr19' 'chr2' 'chr3' 'chr4' 'chr5' 'chr6' 'chr7' 'chr8' 'chr9'
 'chrM' 'chrX' 'chrY']
22


In [6]:
# calculating read lengths
def calculate_read_length(df):
    df['read_length']=df['end']-df['start']
    return df['read_length'].value_counts()

In [7]:
calculate_read_length(esc_df)

read_length
27    5022714
32    1515212
Name: count, dtype: int64

In [8]:
calculate_read_length(npc_df)

read_length
32    4640956
27    3308850
Name: count, dtype: int64

In [9]:
display(esc_df.head())

,chromosome,start,end,orientation,id,mismatch,sequence,read_length
0,chr5,35526975,35527007,-,3481.6.1,1,GCAATAACTTAAGTTCATTATAATCCATTAAA,32
1,chr6,99092111,99092143,+,3481.6.2,1,GAATATGGGAAGCCTGCAGCAACAGGCTCATT,32
2,chr19,49690961,49690993,-,3481.6.3,4,GCTATTGATGAGTGTGTTGAGGGCAACCTAAC,32
3,chr19,56008355,56008387,-,3481.6.4,2,GGTCTCCGCATAGGTATGGCTCACCGCGGTTG,32
4,chr2,121808458,121808490,+,3481.6.5,1,GAGGGTTGGGAACTCACAGGCATCCTTGGGCC,32


 ESC and NPC libraries contain a mixture of 27 bp and 32 bp single-end reads. Read length itself should not define PCR duplicate identity.

### Step 2 - Remove potential PCR duplicates

The libraries were prefered with PCR amplification, which can produce multiple sequencing tags from the same original ChIP fragment.

In order to not to have several identical copies of the same original DNA molecule, Xu et al.(2008) treat tags which are mapped to the same genomic position and within the same orienttion as a single copy.

This reduces the risk of interpreting technical PCR amplification as biological H3K27me3.

In [10]:
def define_tag_position(df):  
    #copying df for not modyfing the original raw dataset
    df = df.copy()
    df["tag_position"] = np.where(df["orientation"]=="-",df["end"]-1,df["start"])
    return df

esc_df=define_tag_position(esc_df)
npc_df=define_tag_position(npc_df)

In [11]:
display(esc_df.head())

,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position
0,chr5,35526975,35527007,-,3481.6.1,1,GCAATAACTTAAGTTCATTATAATCCATTAAA,32,35527006
1,chr6,99092111,99092143,+,3481.6.2,1,GAATATGGGAAGCCTGCAGCAACAGGCTCATT,32,99092111
2,chr19,49690961,49690993,-,3481.6.3,4,GCTATTGATGAGTGTGTTGAGGGCAACCTAAC,32,49690992
3,chr19,56008355,56008387,-,3481.6.4,2,GGTCTCCGCATAGGTATGGCTCACCGCGGTTG,32,56008386
4,chr2,121808458,121808490,+,3481.6.5,1,GAGGGTTGGGAACTCACAGGCATCCTTGGGCC,32,121808458


In [12]:
#removing duplicates who have the same chromosome, tag position and orientation

def remove_duplicates(df):
    df = df.drop_duplicates(subset=['chromosome','orientation','tag_position'],keep='first')
    return df

esc_dedup = remove_duplicates(esc_df)
npc_dedup = remove_duplicates(npc_df)

print("ESC before",esc_df.shape)
print('ESC after',esc_dedup.shape)
print('NPC before',npc_df.shape)
print('NPC after', npc_dedup.shape)

ESC before (6537926, 9)
ESC after (6535027, 9)
NPC before (7949806, 9)
NPC after (7946364, 9)


In [13]:
def dropped(df1,df2):
    n_dropped = len(df1)-len(df2)
    percentage= n_dropped/len(df1) *100
    return n_dropped, percentage

print('Dropped from ESC:',dropped(esc_df,esc_dedup))
print('Dropped from NPC:',dropped(npc_df,npc_dedup))

Dropped from ESC: (2899, 0.04434127887039407)
Dropped from NPC: (3442, 0.043296654031557504)


#### Duplicate-removal summary

After collapsing tags mapped to the same genomic position and orientation:

- **ESC:** 2899 tags were removed, corresponding to  **0.044%** of the aligned reads.

- **NPC:** 3,442 tags were removed, corresponding to  **0.043%** of the aligned reads.

Only a very small fraction of the aligned tags were removed at this preprocessing step.

### Step 3 — Estimate the center of each ChIP fragment with the 100 bp shift

The aligned tag represents only one end of the original ChIP fragment. Xu et a. (2008) assume a median fragment length of approximately 200 bp and estimate the fragment center by shifting tag position by 100 bp in the direction of its orientation.

'+' tags are shifted ** + 100 bp **, '-' tags are shifted ** -100 bp **

The fragment center is more biologically meaningful than the sequenced end because it provides an approximation of where the immunipreipitated chromatin fragment was located

In [14]:
def fragment_center(df):
    df = df.copy()
    df['frag_center']=np.where(df['orientation']=='-',df['tag_position']-100,df['tag_position']+100)
    return df

npc_centered=fragment_center(npc_dedup)
esc_centered=fragment_center(esc_dedup)   

In [15]:
display(esc_centered.head())

,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center
0,chr5,35526975,35527007,-,3481.6.1,1,GCAATAACTTAAGTTCATTATAATCCATTAAA,32,35527006,35526906
1,chr6,99092111,99092143,+,3481.6.2,1,GAATATGGGAAGCCTGCAGCAACAGGCTCATT,32,99092111,99092211
2,chr19,49690961,49690993,-,3481.6.3,4,GCTATTGATGAGTGTGTTGAGGGCAACCTAAC,32,49690992,49690892
3,chr19,56008355,56008387,-,3481.6.4,2,GGTCTCCGCATAGGTATGGCTCACCGCGGTTG,32,56008386,56008286
4,chr2,121808458,121808490,+,3481.6.5,1,GAGGGTTGGGAACTCACAGGCATCCTTGGGCC,32,121808458,121808558


In [16]:
#sanity check
print(esc_centered["frag_center"].min(), esc_centered["frag_center"].max())
print(npc_centered["frag_center"].min(),npc_centered["frag_center"].max())
print((esc_centered["frag_center"]<0).sum())
print((npc_centered['frag_center']<0).sum())

-44 197065868
-50 197064368
1
1


####  Boundary check 
The 100 bp strand-aware shift produced a negative fragment-center coordinate in one read from each library. Maybe that is because a '-' strand was located very closely to the beggining of a chromosome. The ChIPDiff reference paper does not specify how suck chromosome-boundary cases were handled.  We should inspect these reads  before deciding how to treat them.

In [17]:
print('esc_centered negative fragment center')
display(esc_centered[esc_centered['frag_center']<0])
print()
print('npc_centered negative fragment center')
display(npc_centered[npc_centered['frag_center']<0])

esc_centered negative fragment center


,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center
6483479,chrM,30,57,-,2491.7.1382041,0,TACAATTATCCATCTAAGCATTTTCAG,27,56,-44



npc_centered negative fragment center


,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center
3216467,chrM,24,51,-,3099.7.1800207,1,TATCCATCTAAGCCTTTTCAGTGCTTT,27,50,-50


We see that both of the cases occur on chromosome = chrM(mithocondrial chromosome) near the reference-coordinate origin. This is important because mitochondrial DNA its circular, not linear. The hypothesis is that shifting the '-' strand tag by 100 bp crossed the arbitarary linear reference boundary and produced a negative coordinate. 

#### Implementation note — Exclusion of mitochondrial reads

All mitochondrial (`chrM`) reads are excluded from the downstream analysis,
so the analysis is restricted to the nuclear genome.

This also removes the chromosome-boundary cases that produced negative
fragment-center coordinates after the 100 bp shift.

The original ChIPDiff paper does not explicitly describe mitochondrial
chromosome handling, so this exclusion is documented as an implementation
choice.

In [18]:
npc_centered_updated = npc_centered.drop(npc_centered[npc_centered['chromosome']=='chrM'].index)
display(npc_centered_updated[npc_centered_updated['frag_center']<0])
esc_centered_updated = esc_centered.drop(esc_centered[esc_centered['chromosome']=='chrM'].index)
display(esc_centered_updated[esc_centered_updated['frag_center']<0])

,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center


,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center


### Step 4 - Assign fragment centers to 1 kb genomic bins

Xu et al. (2008) partition the genome into 1 kb bins and count the estimated ChIp fragment centers falling within each bin. 
For every bin $i$, we obtain two counts: $ x_{ESC,i}$ and $ x_{NPC,i} $ representing the number of H3K27me3 ChIP fragments observed in that genomic
interval in the ESC and NPC libraries, respectively.

This converts millions of individual reads into two comparable genomic
signal profiles.

For this implementation, bin membership is determined from the estimated fragment-center coordinate using 1000 bp intervals

In [19]:
def assign_bins(df,bin_size=1000):
    df = df.copy()
    df['bin' ]= df['frag_center']//bin_size 
    df['bin_start'] = df['bin'] * bin_size
    df['bin_end'] = df['bin_start'] + bin_size
    return df

esc_binned = assign_bins(esc_centered_updated)
npc_binned = assign_bins(npc_centered_updated)

In [20]:
display(esc_binned.head())

,chromosome,start,end,orientation,id,mismatch,sequence,read_length,tag_position,frag_center,bin,bin_start,bin_end
0,chr5,35526975,35527007,-,3481.6.1,1,GCAATAACTTAAGTTCATTATAATCCATTAAA,32,35527006,35526906,35526,35526000,35527000
1,chr6,99092111,99092143,+,3481.6.2,1,GAATATGGGAAGCCTGCAGCAACAGGCTCATT,32,99092111,99092211,99092,99092000,99093000
2,chr19,49690961,49690993,-,3481.6.3,4,GCTATTGATGAGTGTGTTGAGGGCAACCTAAC,32,49690992,49690892,49690,49690000,49691000
3,chr19,56008355,56008387,-,3481.6.4,2,GGTCTCCGCATAGGTATGGCTCACCGCGGTTG,32,56008386,56008286,56008,56008000,56009000
4,chr2,121808458,121808490,+,3481.6.5,1,GAGGGTTGGGAACTCACAGGCATCCTTGGGCC,32,121808458,121808558,121808,121808000,121809000


#### Step 5 - Combine the ESC and NPC bin-count profiles

After assigning each estimated fragment center to a 1 kb genomic bin, Xu et al. (2008) alignes the two bin-count tables  by genomic position so that every observed bin has both an ESC and an NPC count.

A bin observed in only one library receives a count of zero in the other
library. At this point, each genomic bin can therefore be represented as:  $  (x_{ESC,i}, x_{NPC,i})$

These paired counts form the observations used by the downstream
statistical model.

In [21]:
def create_counts(df,count_name):
    counts = (df.groupby(['chromosome','bin','bin_start','bin_end']).size().reset_index(name=count_name))
    return counts

print("ESC COUNTS : ")
esc_counts = create_counts(esc_binned,"esc_counts")
display(esc_counts.head())
print()
print("NPC COUNTS :")
npc_counts = create_counts(npc_binned,"npc_counts")
display(npc_counts.head())

ESC COUNTS : 


,chromosome,bin,bin_start,bin_end,esc_counts
0,chr1,3002,3002000,3003000,1
1,chr1,3015,3015000,3016000,1
2,chr1,3017,3017000,3018000,2
3,chr1,3018,3018000,3019000,2
4,chr1,3019,3019000,3020000,1



NPC COUNTS :


,chromosome,bin,bin_start,bin_end,npc_counts
0,chr1,3000,3000000,3001000,2
1,chr1,3001,3001000,3002000,1
2,chr1,3002,3002000,3003000,5
3,chr1,3003,3003000,3004000,2
4,chr1,3004,3004000,3005000,1


In [22]:
bin_counts = pd.merge(esc_counts,npc_counts,on=['chromosome','bin','bin_start','bin_end'],how="outer").fillna(0)
bin_counts[['esc_counts','npc_counts']]=bin_counts[['esc_counts','npc_counts']].astype(int)
display(bin_counts.head(10))

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts
0,chr1,3000,3000000,3001000,0,2
1,chr1,3001,3001000,3002000,0,1
2,chr1,3002,3002000,3003000,1,5
3,chr1,3003,3003000,3004000,0,2
4,chr1,3004,3004000,3005000,0,1
5,chr1,3011,3011000,3012000,0,1
6,chr1,3013,3013000,3014000,0,1
7,chr1,3014,3014000,3015000,0,1
8,chr1,3015,3015000,3016000,1,0
9,chr1,3017,3017000,3018000,2,4


In [23]:
#sanity check
print("ESC fragments before aggregiation : ",len(esc_binned))
print("ESC fragments after aggregation : ", bin_counts['esc_counts'].sum())
print()
print("NPC counts before aggregation " ,len(npc_binned))
print("NPC counts after agregation", bin_counts['npc_counts'].sum())

ESC fragments before aggregiation :  6534780
ESC fragments after aggregation :  6534780

NPC counts before aggregation  7945896
NPC counts after agregation 7945896


With our sanity check we verified that every fragment was assigned to exactly one bin. No fragments were lost and no fragments were counted twice during aggregation/merge.

### Step 6- Identify putative histone modification sites

Before asking whether ESC and NPC are different, ChIPDiff first asks 
iff that genomic bin contains enough H3K27me3 signal to be considered a histone-modified site at all.

For each bin, Xu et al. define the sequencing-depth-normalized score
$ F(i) = \frac{x_{ESC,i}}{n_{ESC}} + \frac{x_{NPC,i}}{n_{NPC}} $
where $n_{ESC}$ and $n_{NPC}$ are the total numbers of fragments in the two libraries.

The score  represents the **combined normalized H3K27me3 signal**
in the bin. A bin is retained as a putative histone modification site when $ F(i) > \frac{2}{m\eta} $ where  $m$ is the total number of 1 kb genomic bins and $\eta \approx 0.7 $ is the estimated fraction of the mouse genome that can be reliably interrogated by read mapping.

In [24]:
#  Total sequencing depth of each library

n_ESC = bin_counts["esc_counts"].sum()
print(n_ESC)
n_NPC=bin_counts["npc_counts"].sum()
print(n_NPC)

6534780
7945896


In [25]:
# Calculate F(i) from the previous dataset

bin_counts['F_score'] = (
    bin_counts["esc_counts"]/n_ESC 
    + 
    bin_counts["npc_counts"] / n_NPC 
    )

display(bin_counts.head())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score
0,chr1,3000,3000000,3001000,0,2,2.517023e-07
1,chr1,3001,3001000,3002000,0,1,1.258511e-07
2,chr1,3002,3002000,3003000,1,5,7.822830e-07
3,chr1,3003,3003000,3004000,0,2,2.517023e-07
4,chr1,3004,3004000,3005000,0,1,1.258511e-07


In [26]:
chromosomes = sorted(set(esc_binned['chromosome']) | set(npc_binned['chromosome']) )
print(chromosomes)
print("number of chromosomes :", len(chromosomes))

['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chrX', 'chrY']
number of chromosomes : 21


In [27]:
# Determining the numver of possible 1kb bins using USCS mm8 chromosome-size reference
chrom_sizes = pd.read_csv('../metadata/mm8.chrom.sizes',sep='\t',names=['chromosome','size'])
display(chrom_sizes.head())
print(len(chrom_sizes))

,chromosome,size
0,chr1,197069962
1,chr2,181976762
2,chrX,165556469
3,chr3,159872112
4,chr4,155029701


34


In [28]:
# Retaining only 21 nuclear chromosomes present in ESC or NPC (chr 1- chr 19, chr X, chr Y)
chromosome_sizes= chrom_sizes[chrom_sizes['chromosome'].isin(chromosomes)].copy()
display(chromosome_sizes.head())
print(len(chromosome_sizes))

,chromosome,size
0,chr1,197069962
1,chr2,181976762
2,chrX,165556469
3,chr3,159872112
4,chr4,155029701


21


In [29]:
# Calculate the total number of 1kb bins: m

chromosome_sizes["n_bins"]=np.ceil(chromosome_sizes["size"]/1000).astype(int)
display(chromosome_sizes.head())

m = chromosome_sizes["n_bins"].sum()
print(m)

,chromosome,size,n_bins
0,chr1,197069962,197070
1,chr2,181976762,181977
2,chrX,165556469,165557
3,chr3,159872112,159873
4,chr4,155029701,155030


2644089


In [30]:
# Calculate the chIPDiff threshold

eta= 0.7
threshold = 2/ (eta *m)
print(threshold)

1.0805774151864243e-06


In [31]:
# Identify putative histone modification sites

bin_counts['putative_sites'] = bin_counts['F_score']>threshold
putative_sites = bin_counts[bin_counts['putative_sites']].copy()
putative_sites = putative_sites.reset_index(drop=True)
display(putative_sites.head())
print(len(putative_sites))

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites
0,chr1,3027,3027000,3028000,4,4,0.000001,True
1,chr1,3037,3037000,3038000,3,10,0.000002,True
2,chr1,3061,3061000,3062000,1,11,0.000002,True
3,chr1,3091,3091000,3092000,3,8,0.000001,True
4,chr1,3094,3094000,3095000,3,9,0.000002,True


637045


In [32]:
fraction = len(putative_sites)/m *100
print('Fraction of the possible putative sites on the genome %', fraction.round())

Fraction of the possible putative sites on the genome % 24.0


After applying the ChIPDiff \(F(i)\) threshold, 637,045 1 kb bins, only %24 were retained as putative H3K27me3 modification sites for downstream analysis.

### Step 7- Merge putative sites into histone modification regions

Xu et al. merge consecutive putative modification sites that are within 1 kb of each other into larger histone modification regions.

Here, bins seperated by a genomic gap at the msot 1 kb on the same chromosome are ssigned to the same region. Each 1 kb bin is retained individually and sddigned a 'region_id', since the HMM later analyses the sequence of bins within each regions

In [33]:
putative_sites.columns

Index(['chromosome', 'bin', 'bin_start', 'bin_end', 'esc_counts', 'npc_counts',
       'F_score', 'putative_sites'],
      dtype='str')

In [34]:
#Sort putative sites in genomic order

putative_sites = (putative_sites.sort_values(['chromosome','bin']).reset_index(drop=True))

# Look at the chromosome and end position of the previous putative bin

previous_chromosome = putative_sites['chromosome'].shift(1)
previous_bin_end = putative_sites['bin_end'].shift(1)

#Start a new region when: 1. the chromosome changes or 2. the gap from the previous putative bin is greater than 1 kb
 
new_region = (putative_sites['chromosome'] != previous_chromosome) | (putative_sites['bin_end'] - previous_bin_end > 1000)

# every true value starts a new region, cumsum() gibes all bins in the same region the same region_id

putative_sites['region_id'] = new_region.cumsum().astype(int)

display(putative_sites.head())
print('number of modification regions', putative_sites['region_id'].nunique())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5


number of modification regions 305593


### Step 8 - Estimate modification instensity

Observed ChIP-seq fragment counts are affected by random sampling. ChIPDiff therefore models each bin count with a Binominal distribution and places a Beta prior on the unknown H3K27me3 modification intensity.

For each library, the posterior expected intensity is: $
E[p_{j,i}\mid x_{j,i}] = \frac{\alpha+x_{j,i}} {\alpha+\beta+n_j} $ with $\alpha=1$ and $\beta=m$ as defined by Xu et al. (2008).

In [35]:
alpha = 1
beta = m

In [36]:
putative_sites['esc_intensity'] = ( (alpha + putative_sites['esc_counts']) / (alpha + beta + n_ESC) )
putative_sites['npc_intensity']= ( (alpha + putative_sites['npc_counts'])/ (alpha + beta + n_NPC))
display(putative_sites.head())


,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,npc_intensity
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1,5.447294e-07,4.721442e-07
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2,4.357835e-07,1.038717e-06
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3,2.178917e-07,1.133146e-06
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4,4.357835e-07,8.498595e-07
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5,4.357835e-07,9.442883e-07


Although the ESC and NPC fragment counts are identical in the first bin (4), their estimated modification intensities differ because the NPC library has a greater sequencing depth (\(n_{ESC}=6.53\)M; \(n_{NPC}=7.95\)M).

### Step 9- Compare posterior modification intensities

First way to compare ESC and NPC is to calculate the ratio of their posterior expected H3K27me3 intensities.

Xu et al. define a fold-change threshold $ tau $ , with  $tau$= 3 for the H3K27me3 ESC-NPC analysis. This provides an intuitive first comparaison, but it is not hte final ChIPDiff prediction because fold-change can be unstable at low signal intensities. 

In [37]:
tau = 3 

In [38]:
putative_sites['intensity_ratio'] = putative_sites['esc_intensity'] / putative_sites['npc_intensity']
putative_sites['log_ratio'] = np.log(putative_sites['intensity_ratio'])
display(putative_sites.head())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,npc_intensity,intensity_ratio,log_ratio
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1,5.447294e-07,4.721442e-07,1.153735,0.143005
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2,4.357835e-07,1.038717e-06,0.419540,-0.868596
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3,2.178917e-07,1.133146e-06,0.192289,-1.648755
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4,4.357835e-07,8.498595e-07,0.512771,-0.667925
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5,4.357835e-07,9.442883e-07,0.461494,-0.773286


In [39]:
# fold-change classification using tau

putative_sites['fold_change_state'] = ' non differential'
putative_sites.loc[putative_sites['intensity_ratio'] > tau, 'fold_change_state'] = 'ESC enriched'
putative_sites.loc[putative_sites['intensity_ratio'] < 1/tau, 'fold_change_state'] = 'NPC enriched'

display(putative_sites.head())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,npc_intensity,intensity_ratio,log_ratio,fold_change_state
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1,5.447294e-07,4.721442e-07,1.153735,0.143005,non differential
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2,4.357835e-07,1.038717e-06,0.419540,-0.868596,non differential
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3,2.178917e-07,1.133146e-06,0.192289,-1.648755,NPC enriched
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4,4.357835e-07,8.498595e-07,0.512771,-0.667925,non differential
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5,4.357835e-07,9.442883e-07,0.461494,-0.773286,non differential


In [40]:
print(putative_sites['fold_change_state'].value_counts())

fold_change_state
 non differential    540960
ESC enriched          60666
NPC enriched          35419
Name: count, dtype: int64


Using the simple $tau$ = 3 fold-change, most putative H3K27me3 sites are classified ad non-differential. These labels are provisional, the final ChIPDiff classification is obtained with the HMM.

### Step 10 - Hidden Markov Model

A simple fold-change classification can be unstable when modification intensity is low, because small fragment-count diffrences can produce large variation in the estimated log-ratio.

ChIPDiff addresses this problem by incorporating the spatial correlation between neighbouring genomic bins. Since histone modifications often extend across continous genomic regions, adjacent bins are expected to have related differential states.

The HMM combines:
- Observed ESC/NPC fragment counts in each bin
- Dependance between neighbouring hidden states

In [41]:
# creating posterior distributions: p∣x∼Beta(α+x,β+n−x)

putative_sites["esc_alpha_post"] = (alpha + putative_sites['esc_counts'])
putative_sites['esc_beta_post'] = (beta + n_ESC - putative_sites['esc_counts'])

putative_sites['npc_alpha_post'] = (alpha + putative_sites['npc_counts'])
putative_sites['npc_beta_post'] = (beta + n_NPC - putative_sites['npc_counts'])

In [42]:
display(putative_sites.head())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,npc_intensity,intensity_ratio,log_ratio,fold_change_state,esc_alpha_post,esc_beta_post,npc_alpha_post,npc_beta_post
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1,5.447294e-07,4.721442e-07,1.153735,0.143005,non differential,5,9178865,5,10589981
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2,4.357835e-07,1.038717e-06,0.419540,-0.868596,non differential,4,9178866,11,10589975
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3,2.178917e-07,1.133146e-06,0.192289,-1.648755,NPC enriched,2,9178868,12,10589974
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4,4.357835e-07,8.498595e-07,0.512771,-0.667925,non differential,4,9178866,9,10589977
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5,4.357835e-07,9.442883e-07,0.461494,-0.773286,non differential,4,9178866,10,10589976


#### Step 10B — Compute emission support for one bin

For each hidden state, ChIPDiff evaluates how compatible the observed ESC/NPC fragment counts are with the range of modification intensities allowed by that state.

The emission probability for state \(s_i\) is defined as:

$$
P(x_{ESC,i},x_{NPC,i}\mid s_i)
=
\frac{
\iint_{s_i}
P(x_{ESC,i}\mid p_{ESC,i})
P(x_{NPC,i}\mid p_{NPC,i})
P(p_{ESC,i})
P(p_{NPC,i})
\,dp_{ESC,i}\,dp_{NPC,i}
}{
\iint_{s_i}
P(p_{ESC,i})
P(p_{NPC,i})
\,dp_{ESC,i}\,dp_{NPC,i}
}
$$

which means :
$$
\text{emission}
=
\text{data evidence}
\times
\frac{\text{posterior state mass}}
{\text{prior state mass}}
$$


The three state regions are defined using the fold-change threshold
$tau$=3:

$ \alpha_0: \frac{1}{\tau} \leq \frac{p_{ESC}}{p_{NPC}} \leq \tau $

$ \alpha_1: \frac{p_{ESC}}{p_{NPC}}>\tau $

$ \alpha_2: \frac{p_{ESC}}{p_{NPC}}<\frac{1}{\tau} $

We first calculate the posterior probability mass contained in each ofthese three regions for a single bin.

In [43]:
from scipy.special import betaln
from scipy.integrate import quad
from scipy.stats import beta as beta_distribution



def calculate_state_masses(
    esc_alpha,
    esc_beta,
    npc_alpha,
    npc_beta,
    tau=3
):
    """
    Calculate the posterior probability mass of the three ChIPDiff states
    for one genomic bin.

    A change of variable is used to make the numerical integration stable
    for Beta distributions concentrated very close to zero.
    """

    # Exact change of variable:
    #
    # z = -npc_beta * log(1 - p_NPC)
    #
    # Therefore:
    #
    # p_NPC = 1 - exp(-z / npc_beta)
    #
    # np.expm1(x) computes exp(x) - 1 accurately
    # when x is close to zero.
    def get_npc_intensity(z):

        return -np.expm1(
            -z / npc_beta
        )

    # Probability density of the transformed NPC variable z.
    #
    # Instead of evaluating the original Beta PDF directly near p = 0,
    # evaluate its mathematically equivalent density after the
    # change of variable.
    def npc_transformed_density(z):

        # Special case at z = 0 to avoid log(0).
        if z == 0:

            # If npc_alpha > 1, the density is zero at the boundary.
            if npc_alpha > 1:
                return 0.0

            # If npc_alpha = 1, the p^(alpha-1) term equals 1.
            return np.exp(
                -betaln(npc_alpha, npc_beta)
                - np.log(npc_beta)
            )

        p_npc = get_npc_intensity(z)

        # Compute the density in log-space for numerical stability.
        log_density = (
            (npc_alpha - 1) * np.log(p_npc)
            - z
            - betaln(npc_alpha, npc_beta)
            - np.log(npc_beta)
        )

        return np.exp(log_density)

    # Non-differential condition:
    #
    # p_NPC / tau <= p_ESC <= tau * p_NPC
    def non_differential_integrand(z):

        p_npc = get_npc_intensity(z)

        lower_threshold = p_npc / tau
        upper_threshold = tau * p_npc

        probability_esc_between_thresholds = (
            beta_distribution.cdf(
                upper_threshold,
                esc_alpha,
                esc_beta
            )
            -
            beta_distribution.cdf(
                lower_threshold,
                esc_alpha,
                esc_beta
            )
        )

        return (
            npc_transformed_density(z)
            * probability_esc_between_thresholds
        )

    # ESC-enriched condition:
    #
    # p_ESC > tau * p_NPC
    def esc_enriched_integrand(z):

        p_npc = get_npc_intensity(z)

        esc_threshold = tau * p_npc

        probability_esc_above_threshold = (
            beta_distribution.sf(
                esc_threshold,
                esc_alpha,
                esc_beta
            )
        )

        return (
            npc_transformed_density(z)
            * probability_esc_above_threshold
        )

    # NPC-enriched condition:
    #
    # p_ESC < p_NPC / tau
    def npc_enriched_integrand(z):

        p_npc = get_npc_intensity(z)

        npc_threshold = p_npc / tau

        probability_esc_below_threshold = (
            beta_distribution.cdf(
                npc_threshold,
                esc_alpha,
                esc_beta
            )
        )

        return (
            npc_transformed_density(z)
            * probability_esc_below_threshold
        )

    # Integrate each state directly.
    non_differential_mass = quad(
        non_differential_integrand,
        0,
        np.inf,
        epsabs=1e-12,
        epsrel=1e-10,
        limit=200
    )[0]

    esc_enriched_mass = quad(
        esc_enriched_integrand,
        0,
        np.inf,
        epsabs=1e-12,
        epsrel=1e-10,
        limit=200
    )[0]

    npc_enriched_mass = quad(
        npc_enriched_integrand,
        0,
        np.inf,
        epsabs=1e-12,
        epsrel=1e-10,
        limit=200
    )[0]

    return (
        non_differential_mass,
        esc_enriched_mass,
        npc_enriched_mass
    )

In [44]:
first_bin = putative_sites.iloc[0]

posterior_masses = calculate_state_masses(
    first_bin["esc_alpha_post"],
    first_bin["esc_beta_post"],
    first_bin["npc_alpha_post"],
    first_bin["npc_beta_post"],
    tau
)

print("Posterior state masses:")
print("non-differential:", posterior_masses[0])
print("ESC-enriched:", posterior_masses[1])
print("NPC-enriched:", posterior_masses[2])

Posterior state masses:
non-differential: 0.8946783481651924
ESC-enriched: 0.07384838638139948
NPC-enriched: 0.031473265452561246


For a single genomic bin, the ESC and NPC posterior Beta distributions were used to evaluate how much probability mass falls into each ChIPDiff state region. For the first bin, most of the posterior falls on the non-differential region. These values are not final HMM state probabilities. They are the posterior support for each state before the prior-state normalization requiered by Equation 4.

#### Step 10C - Calculate prior state masses

Equation 4 conditions the emission probability on a hidden state. The proba mass assigned to each state under the Beta prior must also be calculated. 

The same three state regions are used, but before observing the fragment counts: 
$$
p_{ESC} \sim \mathrm{Beta}(\alpha,\beta) 
$$

$$
p_{NPC} \sim \mathrm{Beta}(\alpha,\beta)
$$ 

These prior state masses correspond to the denominator of Equation 4. 


In [45]:
#prior masses so we act like we didn't see the data
# ESC prior/ NPC prior = Beta(alpha, beta)

prior_masses = calculate_state_masses(alpha,beta,alpha,beta,tau)
print("Prior state masses:")
print("non-differential:", prior_masses[0])
print("ESC_enriched:", prior_masses[1])
print("NPC-enriched:", prior_masses[2])

Prior state masses:
non-differential: 0.5000000709128962
ESC_enriched: 0.2499999645435519
NPC-enriched: 0.24999996454355194


The prior distribution assigned equal probability to the ESC enriched and NPC enriched state (%25 each), while about halof the prior mass falls in the non-differential region. 

For the first bin, observing the fragment counts increases the non-differential mass from approximately 0.50 under the prior to 
0.895 under the posterior. 

#### Step 10D- Convert state masses into emission weigths

The posterior state mass alone does not represent the HMM emission, because some state regions alredy recieve more probability under the prior. 

For each state, we compare posterior with the prior. 
$$
w_s
=
\frac{P(s \mid x_{ESC},x_{NPC})}
{P(s)}
$$

These ratios represent the state-dependent part of the ChIPDiff emission probability. A value grater than 1 means that observing the fragment counts increased support for that state relative to the prior. 

In [46]:
emission_weights = (posterior_masses[0]/ prior_masses[0], 
posterior_masses[1] / prior_masses[1], 
posterior_masses[2], prior_masses[2] )

print("Emission Weigths")
print("non_differential", emission_weights[0])
print("ESC enriched", emission_weights [1])
print( "NPC enriched", emission_weights[2])

Emission Weigths
non_differential 1.7893564425534894
ESC enriched 0.2953935874200275
NPC enriched 0.031473265452561246


#### Step 10E- Build an emission lookup table

The emission weights depend on the observed ESC and NPC fragment counts. Many genomic bins have the same pair of fragment counts, so recalculating the numerical integrals for every bin would be unnecessary. 

Following the computational strategy described by Xu et al., we therefore calculate the emission weights once for each unique ESC/NPC count pair and store the results in a lookup table

In [47]:
#seeing how many unique counts are there

unique_count_pairs = putative_sites[['esc_counts','npc_counts']].sort_values(by=['esc_counts','npc_counts']).drop_duplicates().reset_index(drop=True)
print("length of putative sites", len(putative_sites))
print("length of unique count pairs", len(unique_count_pairs))
display(unique_count_pairs.head())


length of putative sites 637045
length of unique count pairs 3709


,esc_counts,npc_counts
0,0,9
1,0,10
2,0,11
3,0,12
4,0,13


In [48]:
def calculate_emission_weights(esc_count, npc_count):
    
    #Calculate posterior beta parameters from the observed count
    esc_alpha_post = alpha + esc_count
    esc_beta_post = beta + n_ESC -esc_count

    npc_alpha_post = alpha + npc_count
    npc_beta_post = beta + n_NPC - npc_count

    #Calculate how much posterior proba mass falls into each of the three ChIPDiff state region
    posterior_masses = calculate_state_masses(esc_alpha_post , esc_beta_post, npc_alpha_post, npc_beta_post)

    # Convert posterior masses to emission weights by dividing each one by its prior state means
    non_dif_emission = posterior_masses[0] / prior_masses[0]
    esc_emission = posterior_masses[1] / prior_masses[1]
    npc_emission = posterior_masses[2] / prior_masses[2]

    return(non_dif_emission, esc_emission, npc_emission)

In [49]:
calculate_emission_weights(4,4)

(1.7893564425534894, 0.2953935874200275, 0.12589307966513075)

In [50]:
#these are not probabilitys, there are relative emission weights

emission_lookup = unique_count_pairs.copy()

emission_results= []

for _ ,row in emission_lookup.iterrows():

    weights = calculate_emission_weights(row['esc_counts'],row['npc_counts'])

    emission_results.append(weights)

# add the calculated emission weights as new colums

emission_lookup[['non_diff_emission','esc_emission','npc_emission']] = emission_results
display(emission_lookup.head())
print(len(emission_lookup))

/var/folders/w5/zdsxlj311w956jpppxpbq67h0000gn/T/ipykernel_22832/3557317990.py:144: IntegrationWarning: The integral is probably divergent, or slowly convergent.
  non_differential_mass = quad(


,esc_counts,npc_counts,non_diff_emission,esc_emission,npc_emission
0,0,9,0.158040,1.093280e-05,3.683910
1,0,10,0.122617,3.036675e-06,3.754763
2,0,11,0.095133,8.434612e-07,3.809734
3,0,12,0.073808,2.342782e-07,3.852383
4,0,13,0.057264,6.507267e-08,3.885473


3709


In [51]:
#Check for infinite values
print('sum of values:', emission_lookup[['non_diff_emission','esc_emission','npc_emission']].sum())

#Check for missing values
print('missing values', emission_lookup[['non_diff_emission','esc_emission','npc_emission']].isna().sum())

#Check for negative values
print('negative values:', (emission_lookup[['non_diff_emission','esc_emission','npc_emission']] < 0).sum())


sum of values: non_diff_emission    3694.217765
esc_emission         6376.293035
npc_emission         1025.999696
dtype: float64
missing values non_diff_emission    0
esc_emission         0
npc_emission         0
dtype: int64
negative values: non_diff_emission    0
esc_emission         0
npc_emission         0
dtype: int64


#### Step 10F- Assign emission weights to all putative bins

The emission lookup table contains one set of relative emission weights for each unique ESC/NPC fragment-count combination.

These precomputed values are now assigned back to all putative genomic bins according to their observed ESC and NPC counts.

In [52]:
putative_sites_with_emissions = putative_sites.merge(emission_lookup,on=['esc_counts','npc_counts'],how='left',validate='many_to_one')
putative_sites_with_emissions = putative_sites_with_emissions.sort_values(['chromosome','bin']).reset_index(drop=True)
display(putative_sites_with_emissions.head())

,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,...,intensity_ratio,log_ratio,fold_change_state,esc_alpha_post,esc_beta_post,npc_alpha_post,npc_beta_post,non_diff_emission,esc_emission,npc_emission
0,chr1,3027,3027000,3028000,4,4,0.000001,True,1,5.447294e-07,...,1.153735,0.143005,non differential,5,9178865,5,10589981,1.789356,0.295394,0.125893
1,chr1,3037,3037000,3038000,3,10,0.000002,True,2,4.357835e-07,...,0.419540,-0.868596,non differential,4,9178866,11,10589975,1.226329,0.000459,1.546883
2,chr1,3061,3061000,3062000,1,11,0.000002,True,3,2.178917e-07,...,0.192289,-1.648755,NPC enriched,2,9178868,12,10589974,0.351023,0.000008,3.297945
3,chr1,3091,3091000,3092000,3,8,0.000001,True,4,4.357835e-07,...,0.512771,-0.667925,non differential,4,9178866,9,10589977,1.451981,0.003666,1.092373
4,chr1,3094,3094000,3095000,3,9,0.000002,True,5,4.357835e-07,...,0.461494,-0.773286,non differential,4,9178866,10,10589976,1.340018,0.001310,1.318655


In [53]:
#sanity check
print('number of bins before merge',len(putative_sites))
print('number of bins after merge',len(putative_sites_with_emissions))
print('negative values:',(putative_sites_with_emissions[['non_diff_emission','esc_emission','npc_emission']]<0).sum())

number of bins before merge 637045
number of bins after merge 637045
negative values: non_diff_emission    0
esc_emission         0
npc_emission         0
dtype: int64


All bins were succesfully matched to their precomputed emission weights with no missing or negative emission values introduced during the merge

### Step 11- Train the HMM transition matrix

Each putative histone-modification region is treated as independent sequence of genomic bins.

The emission weights are fixed, while the transition probabilites between the three hidden states are learned using the Baum-Welch algorithm. Following Xu et al., the H3K27me3 transition matrix will be trained using 10,000 randomly selected modification regions.

In [54]:
#look at the region lenghts
region_lenghts = putative_sites_with_emissions.groupby('region_id').size()
display(region_lenghts)
display(region_lenghts.describe())
print('number of modification regions',len(region_lenghts))

region_id
1         1
2         1
3         1
4         1
5         1
         ..
305589    1
305590    2
305591    2
305592    1
305593    1
Length: 305593, dtype: int64

count    305593.000000
mean          2.084619
std           2.730393
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         179.000000
dtype: float64

number of modification regions 305593


In [55]:
# selection of random 10 000 training regions

all_region_ids = (putative_sites_with_emissions['region_id'].drop_duplicates())
# to_numpy transforms series into array, erases the index number
training_regions = (all_region_ids.sample(n=10000,random_state=42).to_numpy())
display(training_regions)
print('number of training regions', len(training_regions))

array([163129, 251713, 242088, ..., 229889, 240189, 154974],
      shape=(10000,))

number of training regions 10000


#### Initial transition matrix

The HMM contains three hidden states: alpha_0: non-differential, alpha_1: ESC-enriched and alpha_2: NPC-enriched

Before training, the transition probabilities are initialized uniformly. So from any current state the model initially assigns equal probability to transitioning to each of the three possible next states.

$$
A =
\begin{pmatrix}
1/3 & 1/3 & 1/3 \\
1/3 & 1/3 & 1/3 \\
1/3 & 1/3 & 1/3
\end{pmatrix}
$$

These transition probabilities are then updated during Baum–Welch training using the selected histone-modification regions.

In [56]:
#creation of initial uniform transformation matrix

states = ['non differential', 'ESC_enriched', 'NPC_enriched']
transition_matrix = np.full((3,3),1/3)
display(pd.DataFrame(transition_matrix, index = states, columns = states ))

,non differential,ESC_enriched,NPC_enriched
non differential,0.333333,0.333333,0.333333
ESC_enriched,0.333333,0.333333,0.333333
NPC_enriched,0.333333,0.333333,0.333333


### Prepare HMM training sequences

Each selected modification region is represented as an ordered sequence of bins. For every bin, the three precomputed emission weights corresponds to the three hidden states: -non differential - ESC enriched -NPC enriched. A region containing k bins is therefore represented by a k*3 emission matrix

In [57]:
training_bins = putative_sites_with_emissions[putative_sites_with_emissions['region_id'].isin(training_regions)].copy()
training_bins = training_bins.sort_values(['region_id','chromosome','bin'])
print('number of training regions', len(training_bins))
display(training_bins.head())

number of training regions 20741


,chromosome,bin,bin_start,bin_end,esc_counts,npc_counts,F_score,putative_sites,region_id,esc_intensity,...,intensity_ratio,log_ratio,fold_change_state,esc_alpha_post,esc_beta_post,npc_alpha_post,npc_beta_post,non_diff_emission,esc_emission,npc_emission
18,chr1,3206,3206000,3207000,5,12,0.000002,True,17,6.536752e-07,...,0.532493,-0.630185,non differential,6,9178864,13,10589973,1.601967,0.000455,0.795612
19,chr1,3207,3207000,3208000,9,6,0.000002,True,17,1.089459e-06,...,1.648193,0.499680,non differential,10,9178860,7,10589979,1.744281,0.508801,0.002636
20,chr1,3208,3208000,3209000,2,7,0.000001,True,17,3.268376e-07,...,0.432651,-0.837825,non differential,3,9178867,8,10589978,1.206516,0.003622,1.583347
31,chr1,3282,3282000,3283000,2,9,0.000001,True,25,3.268376e-07,...,0.346121,-1.060968,non differential,3,9178867,10,10589976,0.948866,0.000404,2.101864
32,chr1,3283,3283000,3284000,4,5,0.000001,True,25,5.447294e-07,...,0.961446,-0.039317,non differential,5,9178865,6,10589980,1.831143,0.131644,0.206069


In [58]:
#transform each region to numpy array 
emission_columns = ['non_diff_emission', 'esc_emission', 'npc_emission']
training_sequences = []
for region_id, region in training_bins.groupby('region_id',sort= False):
    emission_sequence = region[emission_columns].to_numpy(dtype=float)
    training_sequences.append(emission_sequence)
print('number of training sequences', len(training_sequences))
display(training_sequences[1:5])

number of training sequences 10000


[array([[9.48866292e-01, 4.03554184e-04, 2.10186389e+00],
        [1.83114304e+00, 1.31644289e-01, 2.06069168e-01]]),
 array([[1.6523667 , 0.02668693, 0.66857929]]),
 array([[1.20651574, 0.00362162, 1.58334679],
        [1.55817142, 0.0100282 , 0.87362864]]),
 array([[1.6523667 , 0.02668693, 0.66857929]])]

#### Forward algorithm

For each modification region, the forward algorithm calculates the relative support for each hidden state while moving from the first bin to the last.

The start state S0 is fixed to the non-differential state. Therefore, the state distribution of the first genomic bin is obtained by transitioning from S0 and combining this with first bin's emission weights. 

At each following bin, the perivous state probabilities are propagated through the transition matrix and multiplied by the emission weights of the current bin.

In [59]:
def forward_pass(emission_sequence, transition_matrix):

    n_bins = emission_sequence.shape[0]
    n_states = transition_matrix.shape[0]

    forward_probs = np.zeros((n_bins, n_states))

    scaling_factors = np.zeros(n_bins)

    #S0 is fixed to the non-differential state (state 0)
    # This is why probas of the first genomic bibn are obtained from the first row of the transition matrix
    first_state_probs = transition_matrix[0]

    forward_probs[0] = (first_state_probs * emission_sequence[0])

    #normalize the first row
    scaling_factors[0] = forward_probs[0].sum()
    forward_probs[0] /= scaling_factors[0]
    
    # Move through the remaining bins
    for t in range(1, n_bins):
        #propagate probas from the previous bin through the transition matrix
        predicted_state_probs = (forward_probs[t - 1] @ transition_matrix)
        
        #combine transition info w the emission evidence of the current bin
        # * is element-wise multiplication
        forward_probs[t] = (predicted_state_probs * emission_sequence[t])

        #normalize at each bin for numerical stability at each step
        #now sum of the rows are 1, control for it not to be 0 at long sequences
        scaling_factors[t] = forward_probs[t].sum()
        forward_probs[t] /= scaling_factors[t]

    return forward_probs, scaling_factors

In [60]:
#testing on the first training sequence

forward_probs, scaling_factors = forward_pass(training_sequences[0], transition_matrix)
print('emission sequence:',training_sequences[0])
print('forward probabilities', forward_probs)
print('row sums:', forward_probs.sum(axis=1))

emission sequence: [[1.60196651e+00 4.54636894e-04 7.95611962e-01]
 [1.74428117e+00 5.08800985e-01 2.63625607e-03]
 [1.20651574e+00 3.62161870e-03 1.58334679e+00]]
forward probabilities [[6.68033524e-01 1.89587413e-04 3.31776888e-01]
 [7.73270795e-01 2.25560506e-01 1.16869910e-03]
 [4.31903557e-01 1.29645221e-03 5.66799991e-01]]
row sums: [1. 1. 1.]


### Backward algorithm

The backward algorithm propagates information fron the end of a modification toward its begenning. 

For each hidden state, it evaluates how the observations in the remanining downstream bins are compatible with being in that state.

The backward values are combined with the forward values to obtain the posterior state probabilities for each genomic bin.

In [61]:
def backward_pass(emission_sequence, transition_matrix, scaling_factors):
    #(3,3) -> 3
    n_bins = emission_sequence.shape[0]
    # 3 x 3 -> 3
    n_states = transition_matrix.shape[0]
    # crée une matrice de 3x3 de zéros
    backward_probs = np.zeros((n_bins, n_states))
    # [[0,0,0],[0,0,0][1,1,1]]
    # 1 is a neutral element in multiplication
    # logic: bin0 -> bin 1 -> bin 2 -> nothing
    backward_probs[-1] = 1.0
    # range (3-2,-1,-1) = range(1,-1;-1): 1 0
    # first bin 1 then bin 0, we knoz that bin 2 is all 1
    for t in range(n_bins -2,-1,-1):
        #t=1, emission of next bin * backward_probs[2] = 1 1 1
        next_bin_information = (emission_sequence[t+1]*backward_probs[t+1])
        # matrix multiplication between transition matrix (uniform at first) and next bin info
        backward_probs[t] = (transition_matrix @ next_bin_information)
        #normalisation for it to be coherant avec forward pass
        backward_probs[t] /= scaling_factors[t+1]

    return backward_probs

In [62]:
#testing on the first training sequence
backward_probs = backward_pass(training_sequences[0],transition_matrix,scaling_factors)
print('backward probabilities',backward_probs)
#sum of rows, if we do sum() gives 0 because sums all the matrice
print('row sums',backward_probs.sum(axis=1))

backward probabilities [[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
row sums [3. 3. 3.]


With the initial uniform transition matrix the background algorithm cannot distinguish between the three possible current states using future bins. This is because regardless of the curren state the model intially assines the same probabilirt (1/3) to possible next state

In [63]:
# forward x backword = info from the whole region

posterior_probs = (forward_probs * backward_probs)
posterior_probs = posterior_probs/posterior_probs.sum(axis=1,keepdims=True)
print("posterior states probas",posterior_probs)
print('row sums',posterior_probs.sum(axis=1))

posterior states probas [[6.68033524e-01 1.89587413e-04 3.31776888e-01]
 [7.73270795e-01 2.25560506e-01 1.16869910e-03]
 [4.31903557e-01 1.29645221e-03 5.66799991e-01]]
row sums [1. 1. 1.]


#### Expected transitions between adjacent bins

To update the HMM transition matrix, Baum-Welch requieres the posterior proability of each possible state tranisiton between two neighbouring bins. For each pair of adjecent bins t and t+1, we calculate 
$$
\xi_t(i,j)
=
P(S_t=i,\ S_{t+1}=j \mid \text{whole region})
$$

where i is the state of the current bin and j is the state of the next bin

In [64]:
def calculate_xi(emission_sequence, transition_matrix, forward_probs, backward_probs) :
    n_bins = emission_sequence.shape[0]
    n_states = transition_matrix.shape[0]

    # xi en 3D
    #n_bins -1 : how many transitions are there
    # n_states is fixed to 3 in this case: ND,ESC,NPC
    # 3 x 3 : possible current states x possible next states
    xi = np.zeros((n_bins -1, n_states, n_states))

    # we don't have transition after the last bin so n_bins - 1
    for t in range(n_bins -1):
        transition_scores = (
            #np.newaxis adds a new dimension
            # 3 -> (3,1) column vector
            forward_probs[t][:,np.newaxis]
            * transition_matrix
            * emission_sequence[t+1][np.newaxis, :]
            * backward_probs[t+1][np.newaxis, :]
        )
        #normalising
        xi[t]= transition_scores/ transition_scores.sum()

    return xi

In [65]:
#testing on the first sequence
xi = calculate_xi(training_sequences[0], transition_matrix, forward_probs, backward_probs)
print('xi shape', xi.shape)
print('first transition matrix', xi[0])
print('sum of the first transition matrix',xi[0].sum())

xi shape (2, 3, 3)
first transition matrix [[5.16570815e-01 1.50681980e-01 7.80730177e-04]
 [1.46602410e-04 4.27634327e-05 2.21570639e-07]
 [2.56553378e-01 7.48357626e-02 3.87747350e-04]]
sum of the first transition matrix 1.0000000000000002


#### Baum-Welch transition update

For each training region, the expected transitions are accumulated across the sequence.

Because the initial state S0 is fixed the non-differential state, each region also controbutes an expected transition from S0 to its first hidden state.

The accumulated expected transition counts are then normalized row-wise to obtain an updated transition matrix. 

In [66]:
def update_transition_matrix(training_sequences, transition_matrix):

    n_states = transition_matrix.shape[0]
    transition_counts = np.zeros((n_states,n_states))

    for emission_sequence in training_sequences:

        forward_probs, scaling_factors = forward_pass( emission_sequence,transition_matrix) 
        backward_probs = backward_pass(emission_sequence, transition_matrix, scaling_factors)
        posterior_probs = (forward_probs * backward_probs)
        
        posterior_probs /= posterior_probs.sum(axis=1, keepdims=True)

        #S0 is fixed to the non-differential state (state 0)
        # add the expected transition S0 -> first hidden state
        transition_counts[0] += posterior_probs[0]

        # Add transitions betwen neighboring genomic bins
        if emission_sequence.shape[0] > 1:

            xi = calculate_xi(emission_sequence,transition_matrix, forward_probs, backward_probs)
            transition_counts += xi.sum(axis=0)
        
    updated_transition_matrix = (transition_counts / transition_counts.sum(axis=1, keepdims=True))

    return updated_transition_matrix

In [ ]:
#makes one baum-welch update

updated_transition_matrix = update_transition_matrix(training_sequences,transition_matrix)

display( pd.DataFrame(updated_transition_matrix, index=states, columns=states))

print("Row sums:",updated_transition_matrix.sum(axis=1))

,non differential,ESC_enriched,NPC_enriched
non differential,0.647171,0.189963,0.162866
ESC_enriched,0.538061,0.405300,0.056638
NPC_enriched,0.610446,0.104032,0.285522


Row sums: [1. 1. 1.]


In [70]:
#function witch iteraites on baum-welc matrix till it converges
def train_transition_matrix(training_sequences,initial_transition_matrix, tolerance= 1e-6, max_iterations=100):
    '''
    Train the HMM transition matrix using repeated Baum-Welch updates.
    Training stops when the largest absolute charge in the matrix falls below the convergence tolerance
    '''
    transition_matrix = transition_matrix.copy()
    transition_history=[]
    for iteration in range(max_iterations):
        #complete baum-welch transition update on all selected training regions
        updated_transition_matrix = update_transition_matrix(training_sequences, transition_matrix)
        # measure how much the matrix changed compared to the previous iteration
        #we calculate the absolute difference for every matrix entry and retain the largest one as the convergence criteriation
        max_change = np.max( np.abs(updatet-transition_matrix - transition_matrix))
        transition_history.append(max_change)
        #the updated matrix becomes the starting matrix for next baum-welch iteration
        transition_matrix = updated_transition_matrix
        print(f'iteration{iteration +1}:'f'max transition change ={max_change:.8f}')
        #stop once the transition probas are no longer changing appreciably
        if max_change <tolerance:
            print(f'converged after {iteration +1} iterations')
            break
        return transition_matrix,transition_history

In [71]:
# train the HMM transition matrix using the 10,000 randomly
#selected putative histone-modification regions.
#the initial transition matrix is uniform, as described in ChIPDiff

trained_transition_matrix, transition_history =(train_transition_matrix(training_sequences,transition_matrix))
#display the final transition probabilities in a redable table
display(pd.DataFrame(trained_transition_matrix,index=states,columns=states))

UnboundLocalError: cannot access local variable 'transition_matrix' where it is not associated with a value